# **ENVIROMENT INITIALIZATION**

In [1]:
# IMPORTS
# Math libraries
import numpy as np
import pandas as pd
import polars as pl

# ML libraries
import lightgbm as lgb
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.impute import SimpleImputer

# Technical libraries
from pathlib import Path
from datetime import date, timedelta

In [2]:
# CONFIGURATION
from config import (
    PROJECT_ROOT,
    TRAIN_PATH,
    TARGET_AS_OF,
    DATA_PROCESSED_DIR, DATA_FEATURES_DIR, DATA_SUBMISSIONS_DIR,
    TARGET_FEATURES_PATH,
    MODELS_DIR,
    WINDOWS,
    RANDOM_STATE
)

In [3]:
# DATA LOADING
lf_full = pl.scan_parquet(TRAIN_PATH)

# Filter to final AS_OF
lf_final = lf_full.filter(pl.col("event_date") <= TARGET_AS_OF)

print(f"Data filtered to {TARGET_AS_OF}")

Data filtered to 2026-02-13


# **BUILDING FEATURES FOR TARGET**

In [4]:
# RECENCY FEATURES
recency_features_final = (
    lf_final.group_by("user_id")
    .agg([
        pl.col("event_date").max().alias("last_activity_date"),
        pl.col("event_date").filter(pl.col("to_ord") > 0).max().alias("last_order_date"),
        pl.col("event_date").filter(pl.col("to_cart") > 0).max().alias("last_cart_date"),
        pl.col("event_date").filter(pl.col("search") == 1).max().alias("last_search_date"),
        pl.col("event_date").filter(pl.col("cat") == 1).max().alias("last_cat_date"),
    ])
    .with_columns([
        (pl.lit(TARGET_AS_OF) - pl.col("last_activity_date")).dt.total_days().alias("recency_activity"),
        (pl.lit(TARGET_AS_OF) - pl.col("last_order_date")).dt.total_days().alias("recency_order"),
        (pl.lit(TARGET_AS_OF) - pl.col("last_cart_date")).dt.total_days().alias("recency_cart"),
        (pl.lit(TARGET_AS_OF) - pl.col("last_search_date")).dt.total_days().alias("recency_search"),
        (pl.lit(TARGET_AS_OF) - pl.col("last_cat_date")).dt.total_days().alias("recency_cat"),
    ])
    .drop(["last_activity_date", "last_order_date", "last_cart_date", "last_search_date", "last_cat_date"])
    .collect()
)
recency_features_final = recency_features_final.fill_null(999)

print(f"Recency features shape: {recency_features_final.shape}")

# FREQUENCY FEATURES
frequency_exprs_final = []
for w in WINDOWS:
    window_start = TARGET_AS_OF - timedelta(days = w)
    frequency_exprs_final.extend([
        pl.col("event_date").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).n_unique().alias(f"active_days_{w}d"),
        pl.col("to_ord").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"orders_{w}d"),
        pl.col("to_cart").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"cart_adds_{w}d"),
        pl.col("searches").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"searches_{w}d"),
    ])
frequency_features_final = (
    lf_final.group_by("user_id").agg(frequency_exprs_final).collect()
)

print(f"Frequency features shape: {frequency_features_final.shape}")

# MONETARY FEATURES
monetary_exprs_final = []
for w in WINDOWS:
    window_start = TARGET_AS_OF - timedelta(days = w)
    monetary_exprs_final.extend([
        pl.col("gmv").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"gmv_{w}d"),
        pl.col("gmv_search").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"gmv_search_{w}d"),
        pl.col("gmv_cat").filter(
            (pl.col("event_date") >= window_start) & (pl.col("event_date") <= TARGET_AS_OF)
        ).sum().alias(f"gmv_cat_{w}d"),
    ])
monetary_features_final = (
    lf_final.group_by("user_id").agg(monetary_exprs_final).collect()
)

print(f"Monetary features shape: {monetary_features_final.shape}")

# DAY-OF-WEEK FEATURES
lf_final_dow = lf_final.with_columns(pl.col("event_date").dt.weekday().alias("weekday"))
dow_features_final = (
    lf_final_dow.group_by("user_id")
    .agg([
        pl.col("weekday").filter((pl.col("weekday") == 6) | (pl.col("weekday") == 7)).count().alias("weekend_days"),
        pl.col("to_ord").filter((pl.col("weekday") >= 1) & (pl.col("weekday") <= 5)).sum().alias("weekday_orders"),
        pl.col("to_ord").filter((pl.col("weekday") >= 6) & (pl.col("weekday") <= 7)).sum().alias("weekend_orders"),
        pl.col("gmv").filter(pl.col("weekday") == 2).sum().alias("tuesday_gmv"),
        pl.col("gmv").filter(pl.col("weekday") == 6).sum().alias("saturday_gmv"),
        pl.col("gmv").filter(pl.col("weekday") == 7).sum().alias("sunday_gmv"),
        pl.col("gmv").filter((pl.col("weekday") >= 1) & (pl.col("weekday") <= 5)).sum().alias("weekday_gmv"),
        pl.col("gmv").filter((pl.col("weekday") >= 6) & (pl.col("weekday") <= 7)).sum().alias("weekend_gmv"),
    ])
    .with_columns([
        (pl.col("weekend_orders") / (pl.col("weekend_orders") + pl.col("weekday_orders"))).fill_null(0).alias("weekend_order_share"),
        (pl.col("weekend_gmv") / (pl.col("weekend_gmv") + pl.col("weekday_gmv"))).fill_null(0).alias("weekend_gmv_share"),
    ])
    .collect()
)

print(f"Day-of-week features shape: {dow_features_final.shape}")

# CONVERSION FEATURES
conversion_features_final = (
    lf_final.group_by("user_id")
    .agg([
        pl.col("search_to_cart").sum().alias("search_to_cart_total"),
        pl.col("search_to_ord").sum().alias("search_to_ord_total"),
        pl.col("cat_to_cart").sum().alias("cat_to_cart_total"),
        pl.col("cat_to_ord").sum().alias("cat_to_ord_total"),
        pl.col("searches").sum().alias("searches_total"),
        pl.col("to_cart").sum().alias("cart_adds_total"),
        pl.col("to_ord").sum().alias("orders_total"),
        pl.col("search").sum().alias("search_days_total"),
        pl.col("cat").sum().alias("cat_days_total"),
    ])
    .with_columns([
        (pl.col("search_to_cart_total") / pl.col("searches_total")).fill_null(0).alias("search_to_cart_rate"),
        (pl.col("search_to_ord_total") / pl.col("searches_total")).fill_null(0).alias("search_to_ord_rate"),
        (pl.col("orders_total") / pl.col("cart_adds_total")).fill_null(0).alias("cart_to_ord_rate"),
        (pl.col("cat_to_cart_total") / pl.col("cat_days_total")).fill_null(0).alias("cat_to_cart_rate"),
        (pl.col("cat_to_ord_total") / pl.col("cat_days_total")).fill_null(0).alias("cat_to_ord_rate"),
        (pl.col("orders_total") / pl.col("search_days_total")).fill_null(0).alias("orders_per_active_day"),
        (pl.col("cart_adds_total") / pl.col("search_days_total")).fill_null(0).alias("carts_per_active_day"),
    ])
    .drop([
        "search_to_cart_total", "search_to_ord_total", "cat_to_cart_total", "cat_to_ord_total",
        "searches_total", "cart_adds_total", "orders_total", "search_days_total", "cat_days_total",
    ])
    .collect()
)

print(f"Conversion features shape: {conversion_features_final.shape}")

# SEASONAL FEATURES
lf_final_seasonal = lf_final.with_columns(pl.col("event_date").dt.day().alias("day_of_month"))
seasonal_features_final = (
    lf_final_seasonal.group_by("user_id")
    .agg([
        pl.col("to_ord").filter(pl.col("day_of_month").is_in([10, 11, 25, 26])).sum().alias("salary_day_orders"),
        pl.col("to_ord").filter(~pl.col("day_of_month").is_in([10, 11, 25, 26])).sum().alias("non_salary_day_orders"),
        pl.col("gmv").filter(pl.col("day_of_month").is_in([10, 11, 25, 26])).sum().alias("salary_day_gmv"),
        pl.col("gmv").filter(~pl.col("day_of_month").is_in([10, 11, 25, 26])).sum().alias("non_salary_day_gmv"),
        pl.col("to_ord").sum().alias("total_orders"),
        pl.col("gmv").sum().alias("total_gmv"),
    ])
    .with_columns([
        (pl.col("salary_day_orders") / pl.col("total_orders")).fill_null(0).alias("salary_day_order_share"),
        (pl.col("salary_day_gmv") / pl.col("total_gmv")).fill_null(0).alias("salary_day_gmv_share"),
        (pl.col("salary_day_gmv") / pl.col("salary_day_orders")).fill_null(0).alias("salary_day_aov"),
        (pl.col("non_salary_day_gmv") / pl.col("non_salary_day_orders")).fill_null(0).alias("non_salary_day_aov"),
    ])
    .collect()
)

print(f"Seasonal features shape: {seasonal_features_final.shape}")

# CHANNEL FEATURES
channel_features_final = (
    monetary_features_final.join(frequency_features_final, on = "user_id", how = "inner")
    .with_columns([
        (pl.col("gmv_search_30d") / pl.col("gmv_30d")).fill_null(0).alias("search_gmv_share_30d"),
        (pl.col("gmv_search_90d") / pl.col("gmv_90d")).fill_null(0).alias("search_gmv_share_90d"),
        ((pl.col("gmv_search_30d") > 0) & (pl.col("gmv_cat_30d") > 0)).cast(pl.Int32).alias("uses_both_channels_30d"),
        ((pl.col("gmv_search_90d") > 0) & (pl.col("gmv_cat_90d") > 0)).cast(pl.Int32).alias("uses_both_channels_90d"),
    ])
    .select([
        "user_id",
        "search_gmv_share_30d",
        "search_gmv_share_90d",
        "uses_both_channels_30d",
        "uses_both_channels_90d",
    ])
)

print(f"Channel features shape: {channel_features_final.shape}")

# COMBINE ALL FEATURES
all_features_final = recency_features_final
all_features_final = all_features_final.join(frequency_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(monetary_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(dow_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(conversion_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(seasonal_features_final, on = "user_id", how = "inner")
all_features_final = all_features_final.join(channel_features_final, on = "user_id", how = "inner")

# TREND FEATURES
trend_features_final = (
    frequency_features_final.join(monetary_features_final, on = "user_id", how = "inner")
    .with_columns([
        (pl.col("active_days_7d") / (pl.col("active_days_30d") / 4)).fill_null(1).alias("activity_trend_7_30"),
        (pl.col("active_days_14d") / (pl.col("active_days_30d") / 2)).fill_null(1).alias("activity_trend_14_30"),
        (pl.col("active_days_30d") / (pl.col("active_days_90d") / 3)).fill_null(1).alias("activity_trend_30_90"),
        (pl.col("orders_7d") / (pl.col("orders_30d") / 4)).fill_null(1).alias("orders_trend_7_30"),
        (pl.col("orders_14d") / (pl.col("orders_30d") / 2)).fill_null(1).alias("orders_trend_14_30"),
        (pl.col("orders_30d") / (pl.col("orders_90d") / 3)).fill_null(1).alias("orders_trend_30_90"),
    ])
)
trend_cols = ["user_id"] + [c for c in trend_features_final.columns if "trend" in c]
trend_subset_final = trend_features_final.select(trend_cols)
all_features_final = all_features_final.join(trend_subset_final, on = "user_id", how = "inner")

print(f"Final features shape before cleanup: {all_features_final.shape}")

Recency features shape: (250000, 6)
Frequency features shape: (250000, 17)
Monetary features shape: (250000, 13)
Day-of-week features shape: (250000, 11)
Conversion features shape: (250000, 8)
Seasonal features shape: (250000, 11)
Channel features shape: (250000, 5)
Final features shape before cleanup: (250000, 71)


In [5]:
# REMOVE HIGHLY CORRELATED FEATURES
# Same features as in feature engineering file

cols_to_drop = [
    "total_orders",
    "non_salary_day_orders",
    "weekday_orders",
    "weekend_orders",
    "salary_day_orders",
    "non_salary_day_gmv",
    "weekday_gmv",
    "gmv_search_14d",
    "gmv_search_30d",
    "gmv_search_90d",
]

cols_to_drop_existing = [c for c in cols_to_drop if c in all_features_final.columns]

print(f"Dropping {len(cols_to_drop_existing)} highly correlated features:")
for col in cols_to_drop_existing:
    print(f"  - {col}")

all_features_final = all_features_final.drop(cols_to_drop_existing)

print(f"\nFinal features shape after cleanup: {all_features_final.shape}")
print(f"Number of features (excluding user_id): {len(all_features_final.columns) - 1}")

Dropping 10 highly correlated features:
  - total_orders
  - non_salary_day_orders
  - weekday_orders
  - weekend_orders
  - salary_day_orders
  - non_salary_day_gmv
  - weekday_gmv
  - gmv_search_14d
  - gmv_search_30d
  - gmv_search_90d

Final features shape after cleanup: (250000, 61)
Number of features (excluding user_id): 60


## **CLUSTERING**

In [6]:
# CLUSTERING FOR FINAL PREDICTIONS
# Train KMeans on all data up to TARGET_AS_OF

# Define clustering features
cluster_features = [
    "orders_90d",
    "recency_order",
    "total_gmv",
    "orders_per_active_day",
    "active_days_30d",
    "gmv_90d",
]

print(f"Clustering features: {cluster_features}")

# Extract clustering features
X_cluster_final = all_features_final.select(cluster_features).to_numpy()

print(f"\nBefore cleaning:")
print(f"  Shape: {X_cluster_final.shape}")
print(f"  Has NaN: {np.isnan(X_cluster_final).any()}")
print(f"  Has inf: {np.isinf(X_cluster_final).any()}")
print(f"  Total NaN count: {np.isnan(X_cluster_final).sum()}")

# Replace inf with NaN
X_cluster_final = np.where(np.isinf(X_cluster_final), np.nan, X_cluster_final)

print(f"\nAfter replacing inf with NaN:")
print(f"  Total NaN count: {np.isnan(X_cluster_final).sum()}")

# Impute NaN with 0
imputer_final = SimpleImputer(strategy = "constant", fill_value = 0.0)
X_cluster_final_clean = imputer_final.fit_transform(X_cluster_final)

print(f"\nAfter imputation:")
print(f"  Has NaN: {np.isnan(X_cluster_final_clean).any()}")

# Standardize
scaler_final = StandardScaler()
X_cluster_final_scaled = scaler_final.fit_transform(X_cluster_final_clean)

print(f"\nAfter standardization:")
print(f"  Has NaN: {np.isnan(X_cluster_final_scaled).any()}")

# Fit KMeans and predict
OPTIMAL_K = 4
kmeans_final = KMeans(n_clusters = OPTIMAL_K, random_state = RANDOM_STATE, n_init = 10)
cluster_labels_final = kmeans_final.fit_predict(X_cluster_final_scaled)

print(f"\nCluster distribution:")
unique, counts = np.unique(cluster_labels_final, return_counts = True)
for u, c in zip(unique, counts):
    print(f"  Cluster {u}: {c} users ({c / len(cluster_labels_final) * 100:.2f}%)")

# Add cluster as feature
all_features_final = all_features_final.with_columns(
    pl.Series("cluster", cluster_labels_final).cast(pl.Int32)
)

print(f"\nFinal dataset shape after adding cluster: {all_features_final.shape}")
print(f"Number of features (excluding user_id): {len(all_features_final.columns) - 1}")

Clustering features: ['orders_90d', 'recency_order', 'total_gmv', 'orders_per_active_day', 'active_days_30d', 'gmv_90d']

Before cleaning:
  Shape: (250000, 6)
  Has NaN: True
  Has inf: True
  Total NaN count: 475

After replacing inf with NaN:
  Total NaN count: 528

After imputation:
  Has NaN: False

After standardization:
  Has NaN: False

Cluster distribution:
  Cluster 0: 30832 users (12.33%)
  Cluster 1: 77555 users (31.02%)
  Cluster 2: 135608 users (54.24%)
  Cluster 3: 6005 users (2.40%)

Final dataset shape after adding cluster: (250000, 62)
Number of features (excluding user_id): 61


In [7]:
# FILL NULLS AND SAVE
all_features_final = all_features_final.fill_null(0)

# Check for null values
null_counts = all_features_final.select(pl.all().null_count()).to_pandas().T
null_counts.columns = ["null_count"]
null_counts = null_counts[null_counts["null_count"] > 0]

if len(null_counts) == 0:
    print("\nNo null values in the dataset")
else:
    print(f"\nFound null values in {len(null_counts)} columns:")
    print(null_counts)

print(f"\nFinal features shape: {all_features_final.shape}")
print(f"Number of features (excluding user_id): {len(all_features_final.columns) - 1}")

# Save final features
DATA_FEATURES_DIR.mkdir(parents = True, exist_ok = True)
all_features_final.write_parquet(TARGET_FEATURES_PATH)
print(f"\nFinal features saved to: {TARGET_FEATURES_PATH}")
print(f"File size: {TARGET_FEATURES_PATH.stat().st_size / 1024 / 1024:.2f} MB")


No null values in the dataset

Final features shape: (250000, 62)
Number of features (excluding user_id): 61

Final features saved to: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\processed\features\target_features_2026-02-13.parquet
File size: 32.02 MB


# **MAKING PREDICTIONS**

In [8]:
# LOADING MODELS
clf_model_name = "clf_lightgbm_cv_20260830_233435.txt"
reg_model_name = "reg_lightgbm_cv_20260830_233435.txt"

clf_model_path = MODELS_DIR / "03_lgbm_clf_reg_v2" / clf_model_name
reg_model_path = MODELS_DIR / "03_lgbm_clf_reg_v2" / reg_model_name

if not clf_model_path.exists():
    print(f"ERROR: Classifier model not found: {clf_model_path}")
elif not reg_model_path.exists():
    print(f"ERROR: Regressor model not found: {reg_model_path}")
else:
    print(f"Loading classifier: {clf_model_path}")
    print(f"Loading regressor: {reg_model_path}")
    
    clf_final = lgb.Booster(model_file = str(clf_model_path))
    reg_final = lgb.Booster(model_file = str(reg_model_path))
    
    print("Models loaded successfully!")
    
# MAKING PREDICTIONS WITH OPTIMAL ALPHA
# Optimal alpha from threshold optimization

OPTIMAL_ALPHA = 2.87

# Remove user_id for prediction
feature_cols_final = [c for c in all_features_final.columns if c != "user_id"]
X_final = all_features_final[feature_cols_final].to_numpy()

print(f"Number of features: {len(feature_cols_final)}")
print(f"Number of users: {len(X_final)}")

# Classifier predictions
P_buy_final = clf_final.predict(X_final)

# Regressor predictions
predicted_gmv_log_final = reg_final.predict(X_final)
predicted_gmv_final = np.expm1(predicted_gmv_log_final)
predicted_gmv_final = np.clip(predicted_gmv_final, 0, None)

# Hurdle model with power gate: (P_buy ^ alpha) * predicted_gmv
final_predictions = (P_buy_final ** OPTIMAL_ALPHA) * predicted_gmv_final
final_predictions = np.clip(final_predictions, 0, None)

print(f"\nUsing optimal alpha: {OPTIMAL_ALPHA}")
print(f"P_buy_final range: [{P_buy_final.min():.4f}, {P_buy_final.max():.4f}]")
print(f"predicted_gmv_final range: [{predicted_gmv_final.min():.2f}, {predicted_gmv_final.max():.2f}]")

# CREATING SUBMISSION
submission_df = pd.DataFrame({
    "user_id": all_features_final["user_id"].to_numpy(),
    "predict": final_predictions,
})

# Sort by user_id
submission_df = submission_df.sort_values("user_id")

# Save submission
DATA_SUBMISSIONS_DIR.mkdir(parents = True, exist_ok = True)
submission_path = DATA_SUBMISSIONS_DIR / "submission_02.csv"
submission_df.to_csv(submission_path, index = False)

print(f"\nSubmission saved to: {submission_path}")
print(f"Number of predictions: {len(submission_df)}")
print(f"\nFirst 10 predictions:")
print(submission_df.head(10))

print(f"\nPrediction statistics:")
print(f"  Mean: {final_predictions.mean():.4f}")
print(f"  Median: {np.median(final_predictions):.4f}")
print(f"  Min: {final_predictions.min():.4f}")
print(f"  Max: {final_predictions.max():.4f}")
print(f"  Zeros: {(final_predictions == 0).sum()}")
print(f"  Positive: {(final_predictions > 0).sum()}")

Loading classifier: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\models\03_lgbm_clf_reg_v2\clf_lightgbm_cv_20260830_233435.txt
Loading regressor: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\models\03_lgbm_clf_reg_v2\reg_lightgbm_cv_20260830_233435.txt
Models loaded successfully!
Number of features: 61
Number of users: 250000

Using optimal alpha: 2.87
P_buy_final range: [0.0349, 0.9953]
predicted_gmv_final range: [6.67, 3272.51]

Submission saved to: D:\.workspace\Programming\Projects\E-cup_2026_by_Ozon_Tech\data\submissions\submission_02.csv
Number of predictions: 250000

First 10 predictions:
        user_id     predict
170149        2    0.851688
68349         7   92.017537
25905        15    5.593025
596          18  125.432259
176299       23    0.060842
159464       26    0.048343
29866        27    3.828950
20426        30    0.161863
110469       34    0.277299
116208       37    3.147291

Prediction statistics:
  Mean: 38.3504
  Median: 7.2877
  Mi